# Macroeconomic Impact on Insurance Claims
**Aayush Yagol | Econ-Insurance Correlation Analysis**

## Objective
Quantify the relationship between key Australian macroeconomic indicators and general insurance claims. Specifically:
- Test lead-lag correlations: CPI / WPI / PPI vs claim severity (Gross Claims Incurred)
- Identify the optimal lead time (0–12 months / 0–6 quarters)
- Fit baseline regression models for Loss Ratio forecasting
- Document assumptions, limitations, and stakeholder action triggers

## Data Sources
| Series | Source | Cadence | Code/Sheet |
|--------|--------|---------|------------|
| CPI (Insurance subgroup) | ABS 6401.0 | Monthly | A130393720C |
| WPI (Total hourly excl. bonuses) | ABS 6345.0 | Quarterly | A2603609J |
| PPI (Construction output) | ABS 6427.0 | Quarterly | A2333649T |
| Cash Rate Target | RBA F1.1 | Monthly | FIRMMCRT |
| Gross Claims / GWP / Loss Ratio | APRA GI Stats | Quarterly | Dec-2025 release |

**Note on data depth:** APRA Dec-2025 quarterly publication covers Sep 2023 – Dec 2025 (10 quarters). Full macro history (WPI/PPI/RBA) runs 2015–2026. Correlations are computed on the overlapping window.

In [ ]:
import sys, os
sys.path.insert(0, '..')   # make parent dir importable

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

PROC = '../data/processed/'
print('✅ Imports OK')

---
## 1 — Load & Validate Processed Data

In [ ]:
eco = pd.read_csv(PROC + 'economic_master.csv', parse_dates=['Date'])
ins = pd.read_csv(PROC + 'insurance_master.csv', parse_dates=['Date'])
model_df = pd.read_csv(PROC + 'model_ready.csv', parse_dates=['Date'])

print(f'economic_master : {eco.shape[0]:>4} rows | {eco["Date"].min().date()} → {eco["Date"].max().date()}')
print(f'insurance_master: {ins.shape[0]:>4} rows | {ins["Date"].min().date()} → {ins["Date"].max().date()}')
print(f'model_ready     : {model_df.shape[0]:>4} rows | {model_df["Date"].min().date()} → {model_df["Date"].max().date()}')
print()
print('── Economic master – non-null counts ──')
print(eco[['cpi','wpi','ppi','cash_rate']].notna().sum())
print()
print('── Insurance master preview ──')
display(ins.head())

---
## 2 — Macro Indicator Trends (2015–2026)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle('Australian Macroeconomic Indicators (2015–2026)', fontsize=15, y=1.01)

series = [
    ('wpi',       'Wage Price Index (WPI)',             '#2196F3'),
    ('ppi',       'Producer Price Index – Construction','#4CAF50'),
    ('cash_rate', 'RBA Cash Rate (%)',                  '#FF5722'),
    ('cpi',       'CPI – Insurance Subgroup',           '#9C27B0'),
]

for ax, (col, title, color) in zip(axes, series):
    data = eco.dropna(subset=[col])
    ax.plot(data['Date'], data[col], color=color, lw=2)
    ax.fill_between(data['Date'], data[col], alpha=0.08, color=color)
    ax.set_ylabel(title, fontsize=9)
    ax.set_title(title, fontsize=10, loc='left', pad=3)
    # Mark COVID and rate-hike period
    ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'), alpha=0.07, color='grey', label='COVID')
    ax.axvspan(pd.Timestamp('2022-05-01'), pd.Timestamp('2023-11-01'), alpha=0.07, color='red', label='Rate Hikes')

axes[0].legend(['WPI', 'COVID period', 'RBA rate hike cycle'], fontsize=8, loc='upper left')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../modeling/results/01_macro_trends.png', bbox_inches='tight')
plt.show()
print('Saved → modeling/results/01_macro_trends.png')

---
## 3 — Insurance Claims & Loss Ratio Trends (APRA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('APRA Industry Statistics – Householders & Domestic Motor (2023–2025)', fontsize=13)

# Gross Claims
ax = axes[0, 0]
for col, label, color in [
    ('gross_claims_householders',   'Householders',   '#1565C0'),
    ('gross_claims_domestic_motor', 'Domestic Motor', '#F57F17'),
]:
    if col in ins.columns:
        ax.bar(ins['Date'].astype(str), ins[col] / 1e9, alpha=0.7, label=label)
ax.set_title('Gross Claims Incurred ($B)')
ax.set_ylabel('$AUD Billion')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# GWP
ax = axes[0, 1]
for col, label, color in [
    ('gwp_householders',   'Householders',   '#1565C0'),
    ('gwp_domestic_motor', 'Domestic Motor', '#F57F17'),
]:
    if col in ins.columns:
        ax.bar(ins['Date'].astype(str), ins[col] / 1e9, alpha=0.7, label=label)
ax.set_title('Gross Written Premium ($B)')
ax.set_ylabel('$AUD Billion')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# Loss Ratios
ax = axes[1, 0]
for col, label, color in [
    ('loss_ratio_householders',   'Householders',   '#1565C0'),
    ('loss_ratio_domestic_motor', 'Domestic Motor', '#F57F17'),
]:
    if col in ins.columns:
        ax.plot(ins['Date'], ins[col], marker='o', label=label)
ax.axhline(1.0, color='red', lw=1.2, linestyle='--', alpha=0.6, label='Loss Ratio = 1.0')
ax.set_title('Loss Ratio (Gross Claims / GWP)')
ax.set_ylabel('Ratio')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# YoY Claims growth
ax = axes[1, 1]
for col, label in [
    ('gross_claims_householders_yoy',   'Householders'),
    ('gross_claims_domestic_motor_yoy', 'Domestic Motor'),
]:
    if col in ins.columns:
        ax.bar(ins['Date'].astype(str), ins[col], alpha=0.7, label=label)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Gross Claims YoY Growth (%)')
ax.set_ylabel('% Change')
ax.legend()
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../modeling/results/02_insurance_trends.png', bbox_inches='tight')
plt.show()

---
## 4 — Dual-Axis Overlay: Macro vs Claims

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Macro Indicators vs Insurance Claims – Dual Axis Overlay', fontsize=13)

pairs = [
    ('wpi',       'WPI',       'gross_claims_householders',   'Householders Claims ($B)',   axes[0,0]),
    ('ppi',       'PPI',       'gross_claims_householders',   'Householders Claims ($B)',   axes[0,1]),
    ('cash_rate', 'Cash Rate', 'gross_claims_domestic_motor', 'Domestic Motor Claims ($B)', axes[1,0]),
    ('cpi',       'CPI',       'gross_claims_householders',   'Householders Claims ($B)',   axes[1,1]),
]

for macro_col, macro_label, ins_col, ins_label, ax in pairs:
    ax2 = ax.twinx()
    
    eco_data = eco.dropna(subset=[macro_col])
    ax.plot(eco_data['Date'], eco_data[macro_col], color='#2196F3', lw=2, label=macro_label)
    ax.set_ylabel(macro_label, color='#2196F3', fontsize=9)
    ax.tick_params(axis='y', labelcolor='#2196F3')

    if ins_col in ins.columns:
        ins_data = ins.dropna(subset=[ins_col])
        ax2.scatter(ins_data['Date'], ins_data[ins_col] / 1e9, color='#FF5722', s=60, zorder=5, label=ins_label)
        ax2.plot(ins_data['Date'], ins_data[ins_col] / 1e9, color='#FF5722', lw=1.5, linestyle='--')
        ax2.set_ylabel(ins_label, color='#FF5722', fontsize=9)
        ax2.tick_params(axis='y', labelcolor='#FF5722')

    ax.set_title(f'{macro_label} vs {ins_label}', fontsize=10)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../modeling/results/03_dual_axis_overlay.png', bbox_inches='tight')
plt.show()

---
## 5 — Correlation Matrix

In [ ]:
# Use model_ready (the overlapping merged dataset)
corr_cols = [
    c for c in model_df.columns
    if c not in ['Date', 'quarter']
    and not c.endswith(tuple([f'_lag{i}q' for i in range(1,7)]))
    and not c.endswith('_yoy')
    and not c.startswith('flag_')
    and 'roll' not in c
    and 'volatil' not in c
    and '_yoy' not in c
]
print('Correlation columns:', corr_cols)

corr_data = model_df[corr_cols].dropna(how='all', axis=1)

# Pearson
pearson = corr_data.corr(method='pearson')
# Spearman
spearman = corr_data.corr(method='spearman')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

mask = np.triu(np.ones_like(pearson, dtype=bool))
sns.heatmap(pearson, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax1, mask=mask, vmin=-1, vmax=1, square=True, cbar_kws={'shrink':0.8})
ax1.set_title('Pearson Correlation', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

sns.heatmap(spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax2, mask=mask, vmin=-1, vmax=1, square=True, cbar_kws={'shrink':0.8})
ax2.set_title('Spearman Rank Correlation', fontsize=12)
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Correlation Matrix – Macro vs Insurance (Overlapping Period)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('../modeling/results/04_correlation_matrix.png', bbox_inches='tight')
plt.show()

---
## 6 — Lead-Lag Correlation Heatmap (0–6 Quarters)

In [ ]:
def lag_correlation(macro_series: pd.Series, target_series: pd.Series, max_lag: int = 6):
    """Compute Spearman correlation for macro at t-k vs target at t."""
    results = []
    for lag in range(0, max_lag + 1):
        shifted = macro_series.shift(lag)
        combined = pd.concat([shifted, target_series], axis=1).dropna()
        if len(combined) < 5:
            results.append({'lag_quarters': lag, 'spearman_r': np.nan, 'p_value': np.nan, 'n': 0})
            continue
        r, p = stats.spearmanr(combined.iloc[:, 0], combined.iloc[:, 1])
        results.append({'lag_quarters': lag, 'spearman_r': r, 'p_value': p, 'n': len(combined)})
    return pd.DataFrame(results)

macro_vars   = ['wpi', 'ppi', 'cash_rate', 'cpi']
target_vars  = ['gross_claims_householders', 'gross_claims_domestic_motor',
                'loss_ratio_householders',   'loss_ratio_domestic_motor']

lag_results = {}
for macro in macro_vars:
    for target in target_vars:
        if macro not in model_df.columns or target not in model_df.columns:
            continue
        key = f'{macro} → {target}'
        lag_results[key] = lag_correlation(model_df[macro], model_df[target])

# Build heatmap matrix: rows = lag, cols = macro-target pair
heatmap_r = pd.DataFrame({k: v.set_index('lag_quarters')['spearman_r'] for k, v in lag_results.items()})
heatmap_p = pd.DataFrame({k: v.set_index('lag_quarters')['p_value'] for k, v in lag_results.items()})

fig, ax = plt.subplots(figsize=(max(12, len(heatmap_r.columns) * 1.4), 5))
sig_mask = heatmap_p > 0.1   # mask non-significant
sns.heatmap(
    heatmap_r.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, ax=ax,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Spearman r', 'shrink': 0.8},
)
ax.set_xlabel('Lag (quarters)')
ax.set_ylabel('Macro → Target')
ax.set_title('Lead-Lag Spearman Correlation Heatmap\n(* cells with p < 0.1)', fontsize=12)
plt.tight_layout()
plt.savefig('../modeling/results/05_lag_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('\nTop correlations (any lag):')
top = heatmap_r.abs().max().sort_values(ascending=False)
best_lag = heatmap_r.abs().idxmax()
for pair in top.head(8).index:
    r_val = heatmap_r.loc[best_lag[pair], pair]
    p_val = heatmap_p.loc[best_lag[pair], pair]
    print(f'  {pair:<55} lag={int(best_lag[pair])}Q  r={r_val:+.3f}  p={p_val:.3f}')

---
## 7 — Hypothesis: WPI/PPI Lead Householders Claims

In [ ]:
# Rank correlation significance test across lags
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (macro, label, color) in zip(axes, [
    ('wpi', 'WPI → Householders Claims', '#2196F3'),
    ('ppi', 'PPI → Householders Claims', '#4CAF50'),
]):
    target = 'gross_claims_householders'
    if macro not in model_df.columns or target not in model_df.columns:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue

    res = lag_correlation(model_df[macro], model_df[target], max_lag=6)
    res = res.dropna()

    bars = ax.bar(res['lag_quarters'], res['spearman_r'], color=[
        '#388E3C' if p < 0.1 else '#90A4AE' for p in res['p_value']
    ], alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Lag (quarters)')
    ax.set_ylabel('Spearman r')
    ax.set_title(label, fontsize=11)
    ax.set_ylim(-1, 1)
    ax.set_xticks(res['lag_quarters'])

    # Annotate p-values
    for _, row in res.iterrows():
        sig = '✓' if row['p_value'] < 0.1 else ''
        ax.text(row['lag_quarters'], row['spearman_r'] + 0.03 * np.sign(row['spearman_r']),
                f"{sig}p={row['p_value']:.2f}", ha='center', fontsize=7.5)

    ax.text(0.97, 0.05, '■ p<0.1  □ n.s.', transform=ax.transAxes,
            ha='right', fontsize=8, color='grey')

plt.suptitle('Lead-Lag Rank Correlation: Macro → Householders Claims', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../modeling/results/06_lag_bar_chart.png', bbox_inches='tight')
plt.show()

---
## 8 — Baseline Regression: Loss Ratio ~ Macro Indicators

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

def run_regression(X: pd.DataFrame, y: pd.Series, label: str):
    """Fit OLS and Ridge; return coefficient table + metrics."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    Xs = pd.DataFrame(Xs, columns=X.columns)

    ols = LinearRegression().fit(Xs, y)
    ridge = Ridge(alpha=1.0).fit(Xs, y)

    results = pd.DataFrame({
        'feature':     X.columns,
        'OLS_coef':    ols.coef_,
        'Ridge_coef':  ridge.coef_,
    })

    for name, model in [('OLS', ols), ('Ridge', ridge)]:
        y_hat = model.predict(Xs)
        mae   = mean_absolute_error(y, y_hat)
        rmse  = mean_squared_error(y, y_hat) ** 0.5
        r2    = r2_score(y, y_hat)
        print(f'  [{label}] {name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.3f}')

    return results

# Predictors available in model_ready
feature_candidates = [c for c in model_df.columns if any(c.startswith(p) for p in
    ['wpi', 'ppi', 'cash_rate', 'cpi']) and 'lag' not in c and 'roll' not in c]

targets = [
    ('loss_ratio_householders',   'Householders Loss Ratio'),
    ('loss_ratio_domestic_motor', 'Domestic Motor Loss Ratio'),
]

coef_tables = {}
for target_col, target_name in targets:
    if target_col not in model_df.columns:
        print(f'  ⚠️  {target_col} not in model_ready — skipping')
        continue
    valid_features = [c for c in feature_candidates if c in model_df.columns]
    df_reg = model_df[valid_features + [target_col, 'quarter']].dropna()
    if len(df_reg) < 5:
        print(f'  ⚠️  Insufficient rows for {target_name}')
        continue
    X = df_reg[valid_features + ['quarter']]
    y = df_reg[target_col]
    print(f'\n── {target_name} (n={len(df_reg)}) ──')
    coef_tables[target_col] = run_regression(X, y, target_name)

print('\nCoefficient table:')
for k, v in coef_tables.items():
    print(f'\n{k}:')
    display(v)

---
## 9 — Lag Model: Using Best-Lag Predictors

In [ ]:
# Use lag features (t-1Q, t-2Q) for prediction
lag_features = [c for c in model_df.columns if 'lag' in c and 'yoy' not in c]
print('Available lag features:', lag_features)

for target_col, target_name in targets:
    if target_col not in model_df.columns:
        continue

    valid_lags = [c for c in lag_features if c in model_df.columns]
    df_lag = model_df[valid_lags + [target_col, 'quarter']].dropna()

    if len(df_lag) < 5:
        print(f'  ⚠️  Insufficient rows for lag model: {target_name} (n={len(df_lag)})')
        continue

    X = df_lag[valid_lags + ['quarter']]
    y = df_lag[target_col]

    print(f'\n── Lag model: {target_name} (n={len(df_lag)}) ──')
    run_regression(X, y, f'{target_name} (lag)')

---
## 10 — Macro Index Trend: YoY % Change

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

yoy_cols = [('wpi_yoy', 'WPI YoY %', '#2196F3'), ('ppi_yoy', 'PPI YoY %', '#4CAF50')]
if 'cpi_yoy' in eco.columns:
    yoy_cols.append(('cpi_yoy', 'CPI YoY %', '#9C27B0'))

for col, label, color in yoy_cols:
    data = eco.dropna(subset=[col])
    ax.plot(data['Date'], data[col], lw=2, label=label, color=color)

ax.axhline(0, color='black', lw=0.8)
ax.axhline(3.5, color='orange', lw=1.2, linestyle='--', alpha=0.7, label='CPI warning (3.5%)')
ax.axhline(4.0, color='red',    lw=1.2, linestyle='--', alpha=0.7, label='CPI spike (4.0%)')
ax.fill_between(eco['Date'], 0, eco.get('cpi_yoy', pd.Series(0, index=eco.index)),
                where=eco.get('cpi_yoy', pd.Series(0, index=eco.index)) > 3.5,
                alpha=0.12, color='red', interpolate=True)
ax.set_ylabel('YoY % Change')
ax.set_title('Annual Inflation Rates – WPI, PPI, CPI (2015–2026)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../modeling/results/07_yoy_inflation.png', bbox_inches='tight')
plt.show()

---
## 11 — Key Findings Summary

In [ ]:
from IPython.display import Markdown

findings = """
## Key Findings

### Data Availability
- **WPI / PPI / RBA**: Full history 2015–2026 (11 years, monthly/quarterly)
- **CPI (Insurance subgroup A130393720C)**: Apr 2024 – Feb 2026 only — covers 22 months
- **APRA Industry Stats**: Dec 2025 publication covers Sep 2023 – Dec 2025 (10 quarters)
- **Overlap for correlation**: Sep 2023 – Dec 2025 (10 quarter-end observations)

### Correlation Insights (Spearman, n ≈ 10)
- **WPI** shows [see heatmap] directional relationship with Householders claims — the construction labour cost component is embedded in repair costs
- **PPI (Construction)** tracks closely with WPI; high multicollinearity expected
- **CPI Insurance subgroup** overlaps claims data by only 6 quarters — interpret cautiously
- **Loss Ratio** variation driven primarily by one-off weather events (note Q1 2025 Householders spike = ~$4.7B in claims)

### Regression (n ≈ 10, treat as directional only)
- OLS baseline fits are statistically underpowered with 10 observations
- Ridge regularisation helps stability but overfits are likely
- **Recommendation**: re-run once 2016–2023 historical APRA class-level data is available from legacy GI publications

### Action Triggers (working hypothesis)
| Condition | Suggested Action |
|-----------|------------------|
| WPI > 4% YoY for 2+ consecutive quarters | Flag construction/motor repair cost pressure; review Householders pricing |
| CPI Insurance subgroup > 3.5% YoY | Consider IBNR reserve buffer top-up of +1–1.5% |
| Cash Rate spike >4% in same quarter as PPI acceleration | Stress scenario: stagflation watch |
| Householders Loss Ratio > 0.85 for 2 quarters | Underwriting review trigger |

### Limitations & Caveats
1. **Small-N problem**: 10 quarterly observations is below statistical significance thresholds
2. **Causation vs Correlation**: Rating actions, exposure growth, reinsurance structure, catastrophe events all affect claims independently
3. **CPI subgroup continuity**: Index rebasing post-2020 may affect chain-volume comparability
4. **No policy-level data**: This is industry aggregate only — internal claim frequency and average cost would improve precision
"""

display(Markdown(findings))

# Export to text for reporting
with open('../modeling/results/key_findings.md', 'w') as f:
    f.write(findings)
print('Saved → modeling/results/key_findings.md')

---
*End of notebook – Econ_Insurance_Correlation.ipynb*